In [ ]:
from dataclasses import dataclass, replace
import functools
import pickle
from IPython.display import display
import ipywidgets as widgets
import cvxopt
import numpy as np
from matplotlib import pyplot
import quadprog
from donotation import do
import statemonad
import polymat, polymat.typing
import sosopt, sosopt.typing

# Initialize context object used to create concrete polynomials from polynomial expressions.
# The word context (instead of state) is used in order to not confuse it with the state of a dynamical system.
context = sosopt.init_state()

# Finding Control Barrier Functions (CBFs) and Control Lyapunov Functions (CLFs) for an advanced Safety Filter

This notebook guides the user through the design of an advanced safety filter based on the alternating algorithm.
This involves setting up a sum-of-squares (SOS) optimization problem to find multiple Control Barrier Functions (CBFs) and a single Control Lyapunov-like Function (CLF) for a safety filter applied to a dynamical system.
The CBFs ensure that the system remains within an allowable set $\mathcal X_a$ and operating region $\mathcal X_\text{op}$, whereas the CLF ensures finite-time convergence to the nominal region $\mathcal X_n$.

The code required to run the notebook is structured into three Python packages:
- `statemonad`: Encapsulates stateful operations -- computations that require an explicit state object to execute -- within a monadic structure.
- `polymat`: Provides efficient representation and manipulation of multivariate polynomial matrices.
- `sosopt`: Solves sums-of-squares (SOS) optimization problems.

<!-- This notebook guides you through setting up a sum-of-squares (SOS) optimization problem to find a Control Barrier Function (CBF) for a safety filter applied to a dynamical system.
The CBF ensures that the system remains within an allowable set while respecting permissible ranges for stationary states.

The implementation relies on three Python packages:
- `statemonad`: For representing stateful operations.
- `polymat`: For polynomial matrix manipulations.
- `sosopt`: For solving SOS optimization problems. -->

## Problem setup

In the following, we define the state variables, system model, nominal controller, and constraints required to formulate the CBF and CLF synthesis problem.

### State and Input Variables

First, we define the state and input variables of the dynamical system.
This requires defining each variable using \textit{PolyMat} package.

In [ ]:
# Define a tuple of Python strings representing the state variables of the system model.
# Example: state_variable_names = ('x1', 'x2')
state_variable_names = ...  # [COMPLETE HERE]

# Create symbolic state variables using polymat
state_variables = tuple(polymat.define_variable(name) for name in state_variable_names)
x = polymat.v_stack(state_variables)
n_states = len(state_variables)

# Unpack the state variables into individual Python variables
# Example: x1, x2 = state_variables
... = state_variables  # [COMPLETE HERE]

### System Model

The system dynamics are given by:

$$ \dot{x} = f(x) + G(x)u $$

where:
- $f(x)$ is an $n$-dimensional polynomial vector
- $G(x)$ is an $n \times m$-dimensional polynomial matrix.

In [ ]:
# Define a polymat n-vector representing f(x) using the state variables.
# Example: f = polymat.from_(((x2,), (0,)))
f = ...  # [COMPLETE HERE]

# Define a polymat nxm matrix representing G(x) using the state variables.
# Example: G = polymat.from_(((0,), (1,)))
G = ...  # [COMPLETE HERE]

### Allowable Set and Operating Region

<!-- The allowable set $\mathcal X_a$ defines the safe region where the system should remain. 
It is represented by polynomial inequalities $w(x) \leq 0$.

The operating region $\mathcal X_\text{op}$ defines the permissible range of stationary states considered for this problem. 
It is represented by polynomial inequalities $r(x) \leq 0$, and is selected to contain the allowable set $\mathcal X_a$. -->

The allowable set $\mathcal X_a$ defines the set of states within the system's limit.
It is represented by polynomial inequalities $w(x) \leq 0$.
The operating region $\mathcal X_\text{op}$ defines the set of states for which the safety filter specifications apply.
It is represented by polynomial inequalities $r(x) \leq 0$.

In [ ]:
# Define a tuple of polymat polynomials representing the allowable set X_a.
# Example: w_dict = {'w1': x1**2 - 1, 'w2': x2**2 - 1}
w_dict = ...  # [COMPLETE HERE]

# Define a tuple of polymat polynomials representing the permissible ranges of stationary states.
# Example: op_dict = {}
op_dict = ...  # [COMPLETE HERE]

### Nominal Controller

The nominal controller $u_n(x)$ is a polynomial vector in $x$, representing the legacy control law before applying the safety filter.

In [ ]:
# Define a polymat m-vector representing the nominal controller using the state variables.
# Example: u_n = polymat.from_(-x1 - x2)
u_n = ...  # [COMPLETE HERE]

# modify nominal controller in SOS constraints when no feasible solution can be found
u_n_sos = u_n

### Polynomial Degrees

To formulate the SOS problem, we specify the degrees of the polynomials involved.

In [ ]:
# Define the degrees for the CBF and CLF candidates.
# Example: degree = 2
degree = ...  # [COMPLETE HERE]

## Prerequisites

### CBF and CLF Candidates

We define CBFs for each polynomial in the allowable set `w` and a single CLF to ensure stability and safety. 
The constant coefficient of the CBFs are set to -1 to ensure numerical stability, and derivatives are computed for later use in SOS constraints.

In [ ]:
# Define one CBF for each polynomial in w
B_dict = {}
B_var_dict = {}  # For fixing parameters in the alternating algorithm
for name, expr in w_dict.items():
    context, B_var_dict[name] = sosopt.define_polynomial(
        name=f"B_{name}", 
        monomials=x.combinations(degrees=range(1, degree + 1)),
    ).apply(context)
    B_dict[name] = B_var_dict[name] - 1
dB_dict = {name: B.diff(x).T.cache() for name, B in B_dict.items()}

# Define a single CLF
V_monom = x.combinations(degrees=range(degree + 1))
context, V = sosopt.define_polynomial(
    name="V", 
    monomials=V_monom,
).apply(context)
dV = V.diff(x).T.cache()

### State Feedback Controller

A rational state feedback controller $u(x) = p(x) / s(x)$ is defined to ensure compatibility between CBFs and CLFs.
The numerator `p(x)` is a polynomial vector, and the denominator `s(x)` is a multiplier polynomial scaled to match the degree of the closed-loop system.

In [ ]:
context, (_, n_inputs) = polymat.to_shape(G).apply(context)

# Define the numerator p(x) of the rational controller
context, p = sosopt.define_polynomial(
    name="p", 
    monomials=x.combinations(degrees=range(degree)),
    n_rows=n_inputs,
).apply(context)

G_at_p = (G @ p).cache()

# Compute the degree of the denominator s(x)
context, max_degrees = polymat.to_degree(G_at_p, variables=x).apply(context)

# Define the denominator s(x) as a multiplier
context, s = sosopt.define_multiplier(
    name="s",
    degree=int(np.max(max_degrees)),
    multiplicand=f,
    variables=x,
).apply(context)

# Compute s(x) * f(x)
s_f = (s * f).cache()

# Define closed-loop systems for different controllers
x_dot = (s_f + G_at_p).cache()          # With rational controller u(x) = p(x)/s(x)
x_dot_n = (s_f + G @ u_n).cache()       # With nominal controller u_n
x_dot_n_sos = (s_f + G @ u_n_sos).cache()  # With SOS-based nominal controller (define u_n_sos if needed)

### Operating Region

Two adjustable operating regions are defined — one for CBFs and one for the CLF.
These regions are parameterized with decision variables `delta_opV` and `delta_opB`, initialized to 1 and minimized to 0 during the initialization procedure

In [ ]:
# Define decision variables for operating region adjustment
context, delta_opV = sosopt.define_polynomial(name="delta_opV").apply(context)
context, delta_opB = sosopt.define_polynomial(name="delta_opB").apply(context)

op_dict = w_dict | op_dict

# Define operating regions with adjustable sublevel sets
opV_dict = {name: op + delta_opV for name, op in op_dict.items()}
opB_dict = {name: op + delta_opB for name, op in op_dict.items()}

### Dissipation Rate

The dissipation rate ensures finite-time convergence to the nominal region $\mathcal X_n$. It is set to 0 here but can be adjusted based on system requirements.

In [ ]:
# Define the dissipation rate
dissipation_rate = 0  # Adjust as needed for convergence
# dissipation_rate = 0.01 * s * (V + 1)

## SOS Constraints

This section defines the sum-of-squares (SOS) constraints specified by the advanced safety filter.
These constraints ensure:
- Safety with respect to the safe set $\mathcal X_s$ via CBF conditions.
- Finite-time convergence to the nominal region $\mathcal X_n$ via CLF conditions.
- Containment of the nominal region $\mathcal X_n$ within the safety set $\mathcal X_s$, and the safety set $\mathcal X_s$ within the allowable region $\mathcal X_a$.
- Forward invariance of the nominal region $\mathcal X_n$ under the nominal controller $u_n(x)$.

An initialization procedure will solve the resulting bilinear problem, using margins (`cbf_margin`, `clf_margin`) to ensure improvement of `delta_opV` and `delta_opB` in the consecutive iterations.
When both variables are minimized to 0, an feasible initial point is found for the main alternating algorithm, which then finds the CBF and CLF candidates based on maximizing the volume of $\mathcal X_s$ and $\mathcal X_n$.
<!-- the main alternating algorithm is employed to find the CBF and CLF candidates. -->

In [ ]:
# Margins for solving the bilinear problem using an alternating algorithm
cbf_margin_dict = {}
for name, B in B_dict.items():
    context, cbf_margin_dict[name] = sosopt.define_polynomial(name=f"cbf_margin_{name}").apply(context)
context, clf_margin = sosopt.define_polynomial(name="clf_margin").apply(context)

# Limit CLF/CBF condition margins for numerical stability
max_margin = 0.01

# Initialize tuple of constraints
constraints = tuple()

# Define constraints for each CBF
for name, B in B_dict.items():
    # Control Barrier Function (CBF) condition
    context, cbf_constraint = sosopt.quadratic_module_constraint(
        name=f"cbf_{name}",
        smaller_than_zero=dB_dict[name].T @ x_dot + cbf_margin_dict[name],
        domain=sosopt.set_(
            smaller_than_zero=opB_dict,
            equal_zero={"B": B},
        ),
    ).apply(context)
    constraints = constraints + (cbf_constraint,)

    # CBF margin positivity constraint
    context, cbf_margin_constraint = sosopt.sos_constraint(
        name=f"cbf_margin_max_{name}",
        greater_than_zero=max_margin - cbf_margin_dict[name],
    ).apply(context)
    constraints = constraints + (cbf_margin_constraint,)
    
    # Ensure the safety set contains the nominal region
    context, v_in_b_constraint = sosopt.sos_constraint(
        name=f"v_in_b_{name}",
        greater_than_zero=V - B,
    ).apply(context)
    constraints = constraints + (v_in_b_constraint,)
    
    # Ensure the safety set is contained in the allowable region
    context, b_in_w_constraint = sosopt.quadratic_module_constraint(
        name=f"b_in_w_{name}",
        greater_than_zero=B,
        domain=sosopt.set_(
            greater_than_zero={"w": w_dict[name]},
        ),
    ).apply(context)
    constraints = constraints + (b_in_w_constraint,)

# Control Lyapunov Function (CLF) condition
context, clf_condition = sosopt.quadratic_module_constraint(
    name="clf",
    smaller_than_zero=dV.T @ x_dot + dissipation_rate + clf_margin,
    domain=sosopt.set_(
        greater_than_zero={"V": V},
        smaller_than_zero=opV_dict,
    ),
).apply(context)
constraints = constraints + (clf_condition,)

# CLF margin positivity constraint
context, clf_margin_constraint = sosopt.sos_constraint(
    name="clf_margin_max",
    greater_than_zero=max_margin - clf_margin,
).apply(context)
constraints = constraints + (clf_margin_constraint,)

# Forward invariance condition of the nominal region w.r.t. the nominal controller
context, clf_un_condition = sosopt.quadratic_module_constraint(
    name="clf_un",
    smaller_than_zero=dV.T @ x_dot_n_sos + dissipation_rate,
    domain=sosopt.set_(
        smaller_than_zero=opV_dict,
        equal_zero={"V": V},
    ),
).apply(context)
constraints = constraints + (clf_un_condition,)

# Positivity condition of the denominator
context, s_pos_constraint = sosopt.sos_constraint(
    name="s_pos",
    greater_than_zero=s - 0.001,
).apply(context)
constraints = constraints + (s_pos_constraint,)

# Positivity condition for operating region adjustments
context, delta_opV_constraint = sosopt.sos_constraint(
    name="delta_opV_pos",
    greater_than_zero=delta_opV,
).apply(context)
constraints = constraints + (delta_opV_constraint,)

context, delta_opB_constraint = sosopt.sos_constraint(
    name="delta_opB_pos",
    greater_than_zero=delta_opB,
).apply(context)
constraints = constraints + (delta_opB_constraint,)

# Convert constraints to a dictionary for easier reference
constraints = {c.name: c for c in constraints}

## Pre-Initialization

This section preinitializes the initiatization procedure.
These values can either be computed programmatically or loaded from a pickle file. 
The initial values include:
- The numerator polynomial `p(x)` of the state feedback controller.
- The denominator scalar `s(x)` of the state feedback controller.
- Multipliers introduced through Putinar's Positivstellensatz.
- Operating region adjustments `delta_opV` and `delta_opB`.

These values are stored in a dictionary with the associated symbol of the variable.

In [ ]:
# Import additional required library for file loading
import pickle

# If True, load the initialization from a pickle file
load_from_file = False

if not load_from_file:    
    # Initialize the algorithm with an initial guess
    init_values = (
        (p, u_n),
        (s, 100),
        (constraints['clf'].multipliers["V"], 100),
        (constraints['clf_un'].multipliers["V"], 100),
        *(
            (constraints[f"cbf_{name}"].multipliers["B"], 100) for name in B_dict
        ),
        (delta_opV, 1),
        (delta_opB, 1),
    )

    initial_symbol_values = {}

    for expr, value_expr in init_values:
        if isinstance(value_expr, (float, int)):
            value_expr = polymat.from_vector(value_expr)

        context, result = sosopt.to_symbol_values(expr, value_expr).apply(context)

        initial_symbol_values |= result

else:
    file_name = ...  # [COMPLETE HERE] e.g., 'initial_values.pkl'
    with open(file_name, 'rb') as file:   
        initial_symbol_values = pickle.load(file)

## Alternating Algorithm

This section defines and implements the alternating algorithm to solve the bilinear SOS optimization problem for CBF/CLF synthesis. The algorithm alternates between optimizing different sets of variables (e.g., controller parameters vs. CBF/CLF polynomials) to enforce safety and stability constraints. It includes:
- Data collection for each iteration.
- Definition of optimization steps with linear/quadratic costs and substitutions.
- Solver selection and configuration.
- Two phases: initialization (to reduce operating region margins) and alternation (to maximize the safe set volume).

<!-- ## Initialization Procedure -->
<!-- The initialization procedure generates an initial set of values to initialize the alternating algorithm used to solve the SOS optimization problem. -->

### Data Collected at Each Iteration

We define a data class to store the SDP solver results and the symbol-value results from the previous alternation.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class IterationData:
    symbol_values: dict
    solver_data: sosopt.typing.SolverData | None

iter_data = IterationData(symbol_values=initial_symbol_values, solver_data=None)

### Define Steps/Alternations of the Algorithm

We define a `Step` class that describes each alternation, including costs, substitutions, and constant overrides.
For example, a specific set of variables are substituted by the value of the previous step or set to a constant, resulting in a linear SOS problem.
Multiple steps are tailored to optimize margins, operating regions, and safe set volume.

<!-- to encapsulate each optimization step, including costs, substitutions, and overrides. Multiple steps are tailored to optimize margins, operating regions, and safe set volume. -->

In [ ]:
@dataclass
class Step:
    lin_cost: polymat.typing.MatrixExpression
    quad_cost: polymat.typing.MatrixExpression | None
    substitutions: tuple
    override_symbol_values: dict

def init_step(lin_cost, substitutions, quad_cost=None, override_symbol_values={}):
    return Step(
        lin_cost=lin_cost, quad_cost=quad_cost, 
        substitutions=substitutions, 
        override_symbol_values=override_symbol_values,
    )

# Set CLF/CBF margins to zero when they are not being optimized
set_epsilon_to_zero = {
    clf_margin.symbol: (0,),
    **{cbf_margin.symbol: (0,) for cbf_margin in cbf_margin_dict.values()}
}

step_1_substitutions = (
    p, 
    s,
    constraints['clf'].multipliers["V"],
    constraints['clf_un'].multipliers["V"],
    *(constraints[f'cbf_{name}'].multipliers["B"] for name in B_dict)
)

step_2_substitutions = (
    V,
    *B_var_dict.values()
)

step_1_margin = init_step(
    lin_cost=-clf_margin - sum(cbf_margin_dict.values()),
    substitutions=step_1_substitutions + (
        delta_opV, 
        delta_opB,
    )
)

step_1_opV = init_step(
    lin_cost=delta_opV,
    substitutions=step_1_substitutions + (
        delta_opB, 
        *(constraints['clf'].multipliers[op_name] for op_name in op_dict),
        *(constraints['clf_un'].multipliers[op_name] for op_name in op_dict),
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

step_1_opB = init_step(
    lin_cost=delta_opB,
    substitutions=step_1_substitutions + (
        delta_opV,
        *(constraints[f'cbf_{name}'].multipliers[op_name] for name in B_dict for op_name in op_dict),
        clf_margin,
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

step_2_margin = init_step(
    lin_cost=- clf_margin - sum(cbf_margin_dict.values()),
    substitutions=step_2_substitutions + (
        delta_opV, 
        delta_opB,
    )
)

step_2_opV = init_step(
    lin_cost=delta_opV,
    substitutions=step_2_substitutions + (
        delta_opB,
        *(constraints['clf'].multipliers[op_name] for op_name in op_dict),
        *(constraints['clf_un'].multipliers[op_name] for op_name in op_dict),
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

step_2_opB = init_step(
    lin_cost=delta_opB,
    substitutions=step_2_substitutions + (
        delta_opV,
        *(constraints[f'cbf_{name}'].multipliers[op_name] for name in B_dict for op_name in op_dict),
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

### Select Solver

We select the CVXOPT solver (with an option for MOSEK) and configure its options to suppress progress output and set iteration limits.

In [ ]:
import cvxopt

# Select solver
solver = sosopt.cvxopt_solver
# solver = sosopt.mosek_solver  # Uncomment to use MOSEK instead

# CVXOPT solver options
cvxopt.solvers.options['show_progress'] = False
# cvxopt.solvers.options['show_progress'] = True  # Uncomment for verbose output
# cvxopt.solvers.options['maxiters'] = 100  # Uncomment to limit iterations

### Solve Problem Function

A utility function is defined to solve the SOS problem for each step, updating iteration data with solver results.

In [ ]:
def solve_problem(context, step: Step, iter_data: IterationData):
    problem = sosopt.sos_problem(
        lin_cost=step.lin_cost,
        quad_cost=step.quad_cost,
        constraints=constraints.values(),
        solver=solver,
    )

    # Overwrite values
    symbol_values = iter_data.symbol_values | step.override_symbol_values

    substitutions = {
        symbol: symbol_values[symbol] for param in step.substitutions for symbol in param.iterate_symbols()
    }
    
    # Filter values that need to be substituted
    problem = problem.eval(substitutions)

    # # Uncomment to print decision variables for each constraint
    # for primitive in problem.constraint_primitives:
    #     print(f'{primitive.name=}, {primitive.decision_variable_symbols=}')
    
    # Solve SOS problem
    context, sos_result = problem.solve().apply(context)

    solver_data = sos_result.solver_data
    print(f'{solver_data.status=}, {solver_data.iterations=}, {solver_data.cost=}')
    
    # Update iteration data
    n_iter_data = replace(
        iter_data, 
        symbol_values=symbol_values | sos_result.symbol_values, 
        solver_data=solver_data,
    )
    
    epsilon_roi = (clf_margin, *cbf_margin_dict.values(), delta_opV, delta_opB)
    print(f'epsilon = {tuple(n_iter_data.symbol_values[e.symbol] for e in epsilon_roi)}')

    return context, n_iter_data

### Run Initialization Procedure

The initialization phase reduces the operating region margins (`delta_opV`, `delta_opB`) below a threshold (1e-6) by alternating optimization steps.

In [ ]:
for idx in range(20):
    print(f'iteration: {idx}')
        
    if 1e-6 < iter_data.symbol_values[delta_opB.symbol][0]:
        context, iter_data = solve_problem(context, step_1_margin, iter_data)
        context, iter_data = solve_problem(context, step_1_opB, iter_data)

    if 1e-6 < iter_data.symbol_values[delta_opV.symbol][0]:
        context, iter_data = solve_problem(context, step_2_margin, iter_data)
        context, iter_data = solve_problem(context, step_2_opV, iter_data)
    
    if 1e-6 < iter_data.symbol_values[delta_opV.symbol][0]:
        context, iter_data = solve_problem(context, step_1_margin, iter_data)
        context, iter_data = solve_problem(context, step_1_opV, iter_data)
    
    if 1e-6 < iter_data.symbol_values[delta_opB.symbol][0]:
        context, iter_data = solve_problem(context, step_2_margin, iter_data)
        context, iter_data = solve_problem(context, step_2_opB, iter_data)

    if max(
        iter_data.symbol_values[delta_opB.symbol][0], 
        iter_data.symbol_values[delta_opV.symbol][0]
    ) < 1e-6:
        break

### Run Alternating Algorithm

After initialization, the main algorithm maximizes a surrogate volume of the safe set (via `step_1_vol`) until the primal cost decreases by less than 1%.

In [ ]:
step_1_vol = init_step(
    lin_cost=sosopt.gram_matrix(V, x).trace() + sum(sosopt.gram_matrix(B, x).trace() for B in B_dict.values()),
    quad_cost=sosopt.gram_matrix(V, x).diag() + sum(sosopt.gram_matrix(B, x).diag() for B in B_dict.values()),
    substitutions=step_1_substitutions + (
        delta_opV, 
        delta_opB,
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)


# Maximize a surrogate volume of the safe set until the primal cost decreases by less than 1 percent
previous_cost = None

if not (
    1e-6 < iter_data.symbol_values[delta_opV.symbol][0]
    or 1e-6 < iter_data.symbol_values[delta_opB.symbol][0]
):
    for idx in range(20):
        print(f'iteration: {idx}')
    
        context, iter_data = solve_problem(context, step_2_margin, iter_data)
        context, iter_data = solve_problem(context, step_1_vol, iter_data)

        if previous_cost is not None:
            if (previous_cost - iter_data.solver_data.cost) < 0.005 * iter_data.solver_data.cost:
                break
        previous_cost = iter_data.solver_data.cost

In [ ]:
def save_symbol_values(arg):
    file_name = '2_symbol_values.p'
    
    with open(file_name, 'wb') as file:   
        pickle.dump(iter_data.symbol_values, file)

# Create a button to ensure that the file is not overritten by accident.
button_download = widgets.Button(description = 'Save symbol values')   
button_download.on_click(save_symbol_values)
display(button_download)

## Synthesize Safety Filter

This section synthesizes the advanced safety filter by:
1. Computing state-dependent slack variables $r(x)$ for the CLF and CBF conditions using SOS optimization.
2. Implementing the safety filter as a quadratic program (QP) that adjusts the nominal control input to satisfy safety and stability constraints.

The resulting controller ensures the system remains within the safe set $\mathcal X_s$ defined by the CBFs while converging to the nominal region $\mathcal X_n$ defined by the CLF.

### Find Slack Variables

To synthesize the safety filter, state-dependent slack variables $r(x)$ are computed by solving an SOS problem. These variables relax the CBF and CLF conditions, ensuring feasibility while maintaining safety and stability.
<!-- To synthesize the advanced safety filter, the state dependent slack variables $r(x)$ are found by solving the following SOS problem: -->

In [ ]:
def compute_r(context, h, dh, multiplier, cbf_cond=True):
    constraints_r = []

    context, r = sosopt.define_multiplier(
        name='r',
        degree=-multiplier * h,
        variables=x,
    ).apply(context)
    
    context, upper_bound_constraint = sosopt.quadratic_module_constraint(
        name="upper_bound_constraint",
        smaller_than_zero=r,
        domain=sosopt.set_(
            smaller_than_zero=op_dict,
            equal_zero={h.name: h},
        ) if cbf_cond else sosopt.set_(
            smaller_than_zero=op_dict,
            greater_than_zero={h.name: h},
        ),
    ).apply(context)
    constraints_r.append(upper_bound_constraint)
    
    context, constraint_r_feas = sosopt.quadratic_module_constraint(
        name="constraint_r_feas",
        greater_than_zero=r - dh.T @ x_dot,
        domain=sosopt.set_(
            smaller_than_zero=op_dict | {h.name: h},
            greater_than_zero={"V": V},
        ) if cbf_cond else sosopt.set_(
            smaller_than_zero=op_dict,
            greater_than_zero={"V": V},
        ),
    ).apply(context)
    constraints_r.append(constraint_r_feas)
    
    context, constraint_r_nom = sosopt.quadratic_module_constraint(
        name="constraint_r_nom",
        greater_than_zero=r - dh.T @ x_dot_n_sos,
        domain=sosopt.set_(
            smaller_than_zero={"V": V},
        ),
    ).apply(context)
    constraints_r.append(constraint_r_nom)

    q = (r + multiplier * h).to_linear_coefficients(x)
    
    problem = sosopt.sos_problem(
        lin_cost=r.to_linear_coefficients(x).sum(),
        quad_cost=q,
        constraints=constraints_r,
        solver=solver,
    )
    
    eval_problem = problem.eval(iter_data.symbol_values)
    
    context, sos_result = eval_problem.solve().apply(context)

    solver_data = sos_result.solver_data
    print(f'{solver_data.status=}, {solver_data.iterations=}, {solver_data.cost=}')

    return r.eval(sos_result.symbol_values)

    
rV = compute_r(context, V, dV, constraints['clf'].multipliers['V'], cbf_cond=False)

def gen_rB():
    for name, B in B_dict.items():
        dB = dB_dict[name]
        multiplier = constraints[f'cbf_{name}'].multipliers['B']
    
        yield name, compute_r(context, B, dB, multiplier)

rB_dict = dict(gen_rB())

### Safety Filter Synthesis

The safety filter is implemented as a quadratic program (QP) that adjusts the nominal control input $u_n(x)$ to satisfy the CBF and CLF conditions, using the computed slack variables $r(x)$.

In [ ]:
def create_safety_filter(context, symbol_values):
    context, f_array = polymat.to_array(f, x).apply(context)
    context, G_array = polymat.to_array(G, x).apply(context)
    
    context, V_array = polymat.to_array(V.eval(symbol_values), x).apply(context)
    context, dV_array = polymat.to_array(dV.eval(symbol_values), x).apply(context)
    context, rV_array = polymat.to_array(
        rV, x
    ).apply(context)
    
    B_arrays = []
    dB_arrays = []
    rB_arrays = []
    for name, B in B_dict.items():
        context, array = polymat.to_array(B.eval(symbol_values), x).apply(context)
        B_arrays.append(array)
    
        context, array = polymat.to_array(dB_dict[name].eval(symbol_values), x).apply(context)
        dB_arrays.append(array)
        
        context, array = polymat.to_array(
            rB_dict[name], x
        ).apply(context)
        rB_arrays.append(array)
    
    def controller(x, u_n):        
        fx = f_array(x)[:n_states,:]
        Gx = G_array(x)[:n_states,:]
    
        # x_ = x_val[:n_states,:]
        
        V = V_array(x)
        dV = dV_array(x)
        rV = rV_array(x)
        
        Bs = [array(x) for array in B_arrays]
        dBs = [array(x) for array in dB_arrays]
        rBs = [array(x) for array in rB_arrays]
        
        C = np.vstack((
            -dV.T @ Gx,
            *[-dB.T @ Gx for dB in dBs],
        ))
        b = np.vstack((
            dV.T @ fx - rV,
            *[dB.T @ fx - rB for B, dB, rB in zip(Bs, dBs, rBs)],
        ))

        try:
            u = quadprog.solve_qp(
                G=np.eye(n_inputs),
                a=u_n.reshape(-1),
                C=C.T,
                b=b.reshape(-1),
                meq=0,
            )[0].reshape(-1, 1)
        except:
            # print(f'fall back to u_n at {x}')
            u = u_n
    
        return u
        
    return context, controller

## Plot Results

This section visualizes the results of the CBF/CLF synthesis by plotting:
- A stream plot of the closed-loop system dynamics under the optimized controller.
- Sublevel sets of the Control Lyapunov Function (CLF) `V` and Control Barrier Functions (CBFs) `B_dict`.

The plot is generated for a 2D projection of the state space, assuming the first two dimensions are of interest (e.g., `x1` and `x2`). Adjust the variable labels and limits as needed for your specific system.

In [ ]:
import matplotlib.pyplot as pyplot
import numpy.matlib
import numpy as np

x_min, x_max, y_min, y_max = -1.5, 1.5, -1.5, 1.5

def plot_result(context, symbol_values):
    # Helper function to project the 3-dimensional state onto 2 dimensions
    def map_to_xy(x, y):
        return np.array((x, y) + (0,) * (n_states - 2)).reshape(-1, 1)
    
    pyplot.close()
    fig = pyplot.figure(figsize=(8, 8))
    ax = fig.subplots()

    context, f_array = polymat.to_array(f, x).apply(context)
    context, G_array = polymat.to_array(G, x).apply(context)
    context, u_n_array = polymat.to_array(u_n, x).apply(context)
    context, p_array = polymat.to_array(p.eval(symbol_values), x).apply(context)
    context, s_array = polymat.to_array(s.eval(symbol_values), x).apply(context)
    context, V_array = polymat.to_array(V.eval(symbol_values), x).apply(context)

    B_arrays = []
    for B in B_dict.values():
        context, array = polymat.to_array(B.eval(symbol_values), x).apply(context)
        B_arrays.append(array)

    w_arrays = []
    for w_entry in w_dict.values():
        context, array = polymat.to_array(w_entry.eval(symbol_values), x).apply(context)
        w_arrays.append(array)
        
    # Create stream plot
    ####################

    context, safety_filter = create_safety_filter(context, symbol_values)

    def get_x_dot_rat(x):
        x = np.array(x).reshape(-1, 1)
        u_rat = p_array(x) / s_array(x)
        xdot = f_array(x) + G_array(x) @ u_rat
        return np.squeeze(xdot)
    
    def get_x_dot_n(x):
        x = np.array(x).reshape(-1, 1)
        u_n = u_n_array(x)
        xdot = f_array(x) + G_array(x) @ u_n
        return np.squeeze(xdot)
    
    def get_x_dot(x):
        x = np.array(x).reshape(-1, 1)
        u_n = u_n_array(x)
        u_sf = safety_filter(x, u_n)
        xdot = f_array(x) + G_array(x) @ u_sf
        return np.squeeze(xdot)

    x_step, y_step = (x_max - x_min) / 20, (y_max - y_min) / 20
    ticksX = np.arange(x_min, x_max + x_step, x_step)
    ticksY = np.arange(y_min, y_max + y_step, y_step)
    n_row, n_col = len(ticksY), len(ticksX)
    X = np.matlib.repmat(ticksX, n_row, 1)
    Y = np.matlib.repmat(ticksY.reshape(-1, 1), 1, n_col)
    
    stream_U = np.zeros((n_row, n_col))
    stream_V = np.zeros((n_row, n_col))
    def create_stream_data():
        for row, (x_row, y_row) in enumerate(zip(X, Y)):
            for col, (x, y_val) in enumerate(zip(x_row, y_row)):
                u, v, *_ = get_x_dot(map_to_xy(x, y_val))
                stream_U[row, col] = u
                stream_V[row, col] = v

    all_artists = []
    
    create_stream_data()
    CS = ax.streamplot(X, Y, stream_U, stream_V, density=[0.8, 0.8])

    # Plot Sublevel sets
    ####################
    ticks = np.arange(-2.1, 2.1, 0.04)
    X = np.matlib.repmat(ticks, len(ticks), 1)
    Y = X.T
    
    ZV = np.vectorize(lambda x, y: V_array(map_to_xy(x, y)))(X, Y)
    ax.contour(X, Y, ZV, [0.0], linewidths=2, colors=['#17202A'])

    for array in B_arrays:
        Z = np.vectorize(lambda x, y: array(map_to_xy(x, y)))(X, Y)
        ax.contour(X, Y, Z, [0.0], linewidths=0.5, colors=['#A0B1BA'])
    
    def select_max(arrays):
        def func(x, y):
            def evaluate_arrays():
                for array in arrays:
                    yield array(map_to_xy(x, y))
            return max(evaluate_arrays())
        return func

    Zpick = np.vectorize(select_max(B_arrays))(X, Y)
    CS = ax.contour(X, Y, Zpick, levels=[0], linewidths=2, colors=['#17202A'])
        
    Zpick = np.vectorize(select_max(w_arrays))(X, Y)
    CS = ax.contour(X, Y, Zpick, levels=[0], linewidths=1, colors=['r'], linestyles=['dashed'])
    
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    ax.set_xlabel(r'$x_1$')
    ax.set_ylabel(r'$x_2$')
    
    pyplot.show()
    return context, fig

context, fig = plot_result(context, iter_data.symbol_values)

### Save Figure to PDF File

This section provides an interactive button to save the generated figure as a PDF file, preventing accidental overwrites.

In [ ]:
def save_figure(arg):
    fig.savefig('2_stream_plot.pdf', bbox_inches='tight')

# Create a button to ensure the file is not overwritten by accident
button_download = widgets.Button(description='Save figure')   
button_download.on_click(save_figure)
display(button_download)

## Pickle Results

This subsection saves the results in a pickle file, enabling further simulations and analysis.

In [ ]:
def save_arrays(arg):
    @do()
    def gen_arrays(symbol_values):
        symbol_values = iter_data.symbol_values

        s_array = yield from polymat.to_array(s.eval(symbol_values), x)
        
        V_array = yield from polymat.to_array(V.eval(symbol_values), x)
        B1_array = yield from polymat.to_array(B1.eval(symbol_values), x)
        B2_array = yield from polymat.to_array(B2.eval(symbol_values), x)

        dV_array = yield from polymat.to_array(dV.eval(symbol_values), x)
        dB1_array = yield from polymat.to_array(dB1.eval(symbol_values), x)
        dB2_array = yield from polymat.to_array(dB2.eval(symbol_values), x)

        gV_array = yield from polymat.to_array(constraints['clf'].multipliers['V'].eval(symbol_values), x)
        gB1_array = yield from polymat.to_array(constraints['cbf1'].multipliers['B'].eval(symbol_values), x)
        gB2_array = yield from polymat.to_array(constraints['cbf2'].multipliers['B'].eval(symbol_values), x)

        arrays = {
            's': s_array,
            'V': V_array,
            'B1': B1_array,
            'B2': B2_array,
            'dV': dV_array,
            'dB1': dB1_array,
            'dB2': dB2_array,
            'gV': gV_array,
            'gB1': gB1_array,
            'gB2': gB2_array,
        }

        return statemonad.from_(arrays)

    _, arrays = gen_arrays(iter_data.symbol_values).apply(context)

    file_name = '2_arrays.p'

    with open(file_name, 'wb') as file:   
        pickle.dump(arrays, file)

# Create a button to ensure that the file is not overritten by accident.
button_download = widgets.Button(description = 'Save arrays')   
button_download.on_click(save_arrays)
display(button_download)